# Stack, queue и deque: смысл операций

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv() -> Path:
    for path in (Path("orders_slim.csv"), Path("../../data/orders_slim.csv")):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "orders_slim.csv не найден рядом с ноутбуком или в ../../data/"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from collections import deque


## 1. Поток событий

Возьмите первые 12 заказов. Для каждого сохраните кортеж `(order_id, action)`, где action — `late_review` или `standard`.

In [ ]:
events = None  # TODO: список из 12 кортежей
assert isinstance(events, list) and len(events) == min(12, len(df))
assert all(isinstance(item, tuple) and len(item) == 2 for item in events)
assert {item[1] for item in events} <= {"late_review", "standard"}
print(events[:3])


## 2. Stack: последнее действие отменяется первым

Добавьте четыре шага подготовки в список `history`. Снимите последний шаг методом `pop()` в `undone`.

In [ ]:
steps = ["load", "drop_missing", "make_delay", "scale"]
history = []  # TODO
undone = None  # TODO
assert history == steps[:-1]
assert undone == "scale"
print("осталось:", history, "| отменено:", undone)


## 3. Функция undo

Реализуйте `undo(stack)`: пустой stack возвращает `None`, непустой — удаляет и возвращает последний элемент.

In [ ]:
def undo(stack):
    # TODO
    ...


trial = ["load", "filter"]
removed = undo(trial)
assert removed == "filter" and trial == ["load"]
assert undo([]) is None


## 4. Queue: первый пришёл — первый обработан

Переложите ids событий в `deque`, затем обработайте три через `popleft()`. Порядок должен совпасть с входом.

In [ ]:
queue = deque()  # TODO
processed = []  # TODO
assert len(processed) == min(3, len(events))
assert processed == [item[0] for item in events[:3]]
assert len(queue) == len(events) - len(processed)
print(processed)


## 5. Deque: срочное событие в начало

Создайте очередь из обычных ids. Один late-id добавьте слева методом `appendleft`; он должен стать следующим.

In [ ]:
normal_ids = df.loc[df["is_late"].eq(0), "order_id"].head(5).tolist()
late_id = df.loc[df["is_late"].eq(1), "order_id"].iloc[0]
dispatch = None  # TODO: deque
next_id = None  # TODO: снять слева
assert isinstance(dispatch, deque)
assert next_id == late_id
assert list(dispatch) == normal_ids


## 6. Ограниченный буфер

`deque(maxlen=5)` хранит только пять последних событий. Пропустите через него все ids и сохраните снимок.

In [ ]:
recent = deque(maxlen=5)
# TODO: append каждого order_id
snapshot = None  # TODO: обычный list
assert snapshot == df["order_id"].tail(min(5, len(df))).tolist()
assert recent.maxlen == 5
print(snapshot)


## 7. Выбор структуры по операции

Заполните решения ровно словами `stack`, `queue`, `deque`: отмена шага; честная обработка; срочное добавление с двух концов.

In [ ]:
choices = {
    "undo_preprocessing": None,  # TODO
    "fifo_orders": None,         # TODO
    "urgent_both_ends": None,    # TODO
}
assert choices == {
    "undo_preprocessing": "stack",
    "fifo_orders": "queue",
    "urgent_both_ends": "deque",
}


## 8. Эксперимент: нарушенная семантика

Обработайте одни события слева и справа. В `ORDER_NOTE` объясните, какой вариант сохраняет порядок поступления.

In [ ]:
q_left = deque(item[0] for item in events)
q_right = deque(item[0] for item in events)
left_order = []   # TODO
right_order = []  # TODO
ORDER_NOTE = ""   # TODO: не менее 80 символов
assert left_order == [item[0] for item in events]
assert right_order == list(reversed(left_order))
assert len(ORDER_NOTE) >= 80
print(ORDER_NOTE)


## 9. Самостоятельно: журнал с двумя undo

Выполните пять шагов, отмените два и верните словарь `audit` с оставшейся историей и порядком отмены.

In [ ]:
pipeline = ["load", "clean", "join", "scale", "cluster"]
audit = None  # TODO
assert isinstance(audit, dict)
assert audit["remaining"] == pipeline[:3]
assert audit["undone"] == ["cluster", "scale"]
